# IPC to BNS: Full Pipeline Runner

This notebook follows the exact sequence in `README.md`:
1. Install required packages
2. Run `database.py`
3. Run `ingest.py`
4. Run `mapping_creator.py`
5. Start API (`uvicorn`) and UI (`streamlit`)


In [ ]:
# Optional: ensure we are in the project root
from pathlib import Path
import os

project_root = Path.cwd()
print("Current working directory:", project_root)
print("Files:", sorted([p.name for p in project_root.iterdir()])[:20])

In [ ]:
# Install packages from README.md
%pip install fastapi uvicorn sqlalchemy beautifulsoup4 requests pdfminer.six spacy scrapy scikit-learn pypdf2

In [ ]:
# Run setup scripts in order: database -> ingest -> mapping creator
import subprocess
import sys

scripts = ["database.py", "ingest.py", "mapping_creator.py"]

for script in scripts:
    print(f"\n===== Running {script} =====")
    result = subprocess.run([sys.executable, script], text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"{script} failed with exit code {result.returncode}")
    print(f"{script} completed successfully.")

print("\nAll setup scripts completed.")

In [ ]:
# Start FastAPI and Streamlit in background processes
# Run this cell once. Keep kernel alive while services are running.
import subprocess
import sys
from pathlib import Path

root = Path.cwd()
api_log = open(root / "api.log", "w", encoding="utf-8")
ui_log = open(root / "ui.log", "w", encoding="utf-8")

api_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "api:app", "--reload"],
    stdout=api_log,
    stderr=subprocess.STDOUT,
)

ui_proc = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "ui.py"],
    stdout=ui_log,
    stderr=subprocess.STDOUT,
)

print("API PID:", api_proc.pid)
print("UI PID:", ui_proc.pid)
print("API log: api.log")
print("UI log: ui.log")
print("\nOpen UI in browser using the Streamlit URL from ui.log")

In [ ]:
# Optional: stop background services (run if needed)
for name in ["api_proc", "ui_proc"]:
    proc = globals().get(name)
    if proc and proc.poll() is None:
        proc.terminate()
        print(f"Terminated {name} (PID {proc.pid})")
    else:
        print(f"{name} is not running")